Changes V3: Check if degree was calculated right. Add traffic volume for prediction objective.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from generator import RoadNetwork
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm

from models import StructuralRoadEncoder
import json

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

### Train Model 

In [ ]:
network = RoadNetwork()
network.load("../../osm_data/porto")

In [ ]:
file_path = "./sre_traindata/"
file_name = "sre_traindata.json"
path = os.path.join(file_path, file_name)

# Open Train Data
with open(path, "r") as fp:
        data_full = np.array(json.load(fp))

In [ ]:
u, c = np.unique(data_full[:,:2], return_counts=True, axis=0)
(c > 1).sum()

In [ ]:
u, c = np.unique(data_full, return_counts=True, axis=0)
(c > 1).sum()

In [ ]:
## REMOVE DUPLICATED DATA

# In the negative selection we sometimes get positive pairs. 
# Thus remove the negative ones
u, c = np.unique(data_full[:,:2], return_counts=True, axis=0)
duplicates = u[c > 1]



In [ ]:
duplicates.shape

In [ ]:
# This shows that we have duplicates, with ones positive for geo-locality and one negative.
for i in range(5):
    a = duplicates[i,0]
    b = duplicates[i,1]
    row = data_full[np.where((data_full[:,0] == a) * (data_full[:,1] == b) )]
    print(row)
    

In [ ]:
duplicates

In [ ]:
data_cleaned = []
for i in tqdm(range(len(data_full))):
    row = data_full[i]
    #print(row)
    
    
       
    # Only check those who are negative
    if(row[2] == 0):
        # Now check if this row is in duplicates
        if(any((duplicates[:]==row[:2]).all(1))):
            print(row)
        else:
            data_cleaned.append(row)
    else:
        data_cleaned.append(row)
            
data_cleaned = np.vstack(data_cleaned)

In [ ]:
from joblib import Parallel, delayed

def check_for_duplicates(row, duplicates):
    # Only check those who are negative
    if(row[2] == 0):
        # Now check if this row is in duplicates
        if(any((duplicates[:]==row[:2]).all(1))):
            return None
        else:
            return row
    else:
        return row


res = Parallel(n_jobs=30)(delayed(check_for_duplicates)(data_full[i], duplicates) for i in range(len(data_full)))


In [ ]:
res = [i for i in res if i is not None]
data_cleaned = np.vstack(res)

In [ ]:
print(data_full.shape)
print(data_cleaned.shape)

In [ ]:
"""
Build and train model
"""

network = RoadNetwork()
network.load("../../osm_data/porto")
model = StructuralRoadEncoder(data_cleaned, device, network, out_dim = 4)

In [ ]:
model.train(epochs=10, batch_size=256)

In [ ]:
model.save_model(path="../model_states/sre/")